# detach-clone-snapshot — faded example 3: Complete the per-step gradient snapshot

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-clone-snapshot`. Running the beacon reports progress on the `PyTorch: detach + clone snapshot` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: detach + clone snapshot` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-clone-snapshot`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-clone-snapshot"
DD_SUBTOPIC = "PyTorch: detach + clone snapshot"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A parameter's `.grad` is overwritten or zeroed every iteration, so appending `p.grad` directly aliases the final gradient. Snapshotting with `p.grad.detach().clone()` freezes a graph-free, independently-stored copy of the gradient at that step, making the recorded history honest.

## Faded exercise 3

### Complete the gradient-history snapshot

The loop minimizes a quadratic in a length-3 vector and should record the gradient at each step into `grad_hist`. Complete the line that appends a frozen, non-aliasing copy of the current gradient, so the recorded gradient norms decrease across steps instead of all being the final (near-zero) value.

**Fill in:** Appends a graph-free, independently-stored copy of the current gradient p.grad to the history list.

In [ ]:
t.manual_seed(0)

def grad_history(steps=4, lr=0.1):
    target = t.tensor([1.0, -2.0, 0.5])
    p = t.zeros(3, requires_grad=True)
    grad_hist = []
    for _ in range(steps):
        loss = ((p - target) ** 2).sum()
        loss.backward()
        raise NotImplementedError()  # TODO: append a frozen copy of p.grad to grad_hist
        with t.no_grad():
            p -= lr * p.grad
            p.grad.zero_()
    return t.stack(grad_hist)

print(grad_history().norm(dim=1).tolist())

def _test():
    hist = grad_history()
    assert hist.shape == (4, 3), hist.shape
    # ground-truth: grad of sum((p-target)^2) is 2*(p-target); p stepped by lr*grad
    target = t.tensor([1.0, -2.0, 0.5])
    p = t.zeros(3)
    expected = []
    for _ in range(4):
        g = 2 * (p - target)
        expected.append(g.clone())
        p = p - 0.1 * g
    exp = t.stack(expected)
    assert t.allclose(hist, exp, atol=1e-5), (hist.tolist(), exp.tolist())
    norms = hist.norm(dim=1)
    assert bool((norms[1:] < norms[:-1]).all()), norms.tolist()
    assert not hist.requires_grad

try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
t.manual_seed(0)

def grad_history(steps=4, lr=0.1):
    target = t.tensor([1.0, -2.0, 0.5])
    p = t.zeros(3, requires_grad=True)
    grad_hist = []
    for _ in range(steps):
        loss = ((p - target) ** 2).sum()
        loss.backward()
        grad_hist.append(p.grad.detach().clone())
        with t.no_grad():
            p -= lr * p.grad
            p.grad.zero_()
    return t.stack(grad_hist)

print(grad_history().norm(dim=1).tolist())
```
</details>